# 04 — Feature Engineering
**QM640 Data Analytics Capstone | Walsh College**

Transforms the cleaned dataset into model-ready features:
1. **One-hot encoding** of notes and accords (binary indicators)
2. **Label encoding** of ordinal perception variables (longevity, sillage, season)
3. **Min-max normalisation** of continuous variables
4. **Brand tier classification** (Niche / Luxury / Mass-market)
5. **Release-era cohort** binning (Pre-1990 / 1990s / 2000s / 2010s / 2020s)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path

df = pd.read_csv("../data/processed/fragrantica_clean.csv")
print(f"Input shape: {df.shape}")

## 4.1 One-Hot Encode Notes & Accords

In [ ]:
def explode_pipe_column(df, col, prefix, top_n=50):
    """One-hot encode a pipe-separated column, keeping only top_n values."""
    all_vals = []
    for row in df[col].fillna(""):
        all_vals.extend([v.strip() for v in row.split("|") if v.strip()])
    top_vals = pd.Series(all_vals).value_counts().head(top_n).index.tolist()
    for val in top_vals:
        safe_name = val.replace(" ", "_").replace("/", "_").lower()
        df[f"{prefix}_{safe_name}"] = df[col].fillna("").apply(
            lambda x: int(val in [v.strip() for v in x.split("|")])
        )
    return df

df = explode_pipe_column(df, "top_notes",   prefix="top",    top_n=50)
df = explode_pipe_column(df, "heart_notes", prefix="heart",  top_n=50)
df = explode_pipe_column(df, "base_notes",  prefix="base",   top_n=50)
df = explode_pipe_column(df, "main_accords",prefix="accord", top_n=20)

print(f"Shape after note encoding: {df.shape}")

## 4.2 Encode Concentration & Gender

In [ ]:
conc_dummies = pd.get_dummies(df["concentration"], prefix="conc", drop_first=False)
gender_dummies = pd.get_dummies(df["gender_label"], prefix="gender", drop_first=False)
df = pd.concat([df, conc_dummies, gender_dummies], axis=1)

## 4.3 Release-Era Cohort

In [ ]:
bins  = [0, 1989, 1999, 2009, 2019, 2030]
labels = ["Pre-1990", "1990s", "2000s", "2010s", "2020s"]
df["era_cohort"] = pd.cut(df["release_year"], bins=bins, labels=labels)
era_dummies = pd.get_dummies(df["era_cohort"], prefix="era", drop_first=False)
df = pd.concat([df, era_dummies], axis=1)
print(df["era_cohort"].value_counts())

## 4.4 Normalise Continuous Variables

In [ ]:
scaler = MinMaxScaler()
cont_cols = ["num_votes", "release_year"]
df[[f"{c}_norm" for c in cont_cols]] = scaler.fit_transform(df[cont_cols].fillna(0))
print("Normalisation complete.")

## 4.5 Save Feature Matrix

In [ ]:
OUT_PATH = Path("../data/processed/fragrantica_features.csv")
df.to_csv(OUT_PATH, index=False)
print(f"Feature matrix saved → {OUT_PATH}")
print(f"Final shape: {df.shape}")